# 📬 BERTopic Email Analysis

**Objective:** Discover latent topics across recent emails using BERTopic — a modern topic modeling approach leveraging transformer embeddings, UMAP dimensionality reduction, and HDBSCAN clustering.

**Dataset:** `data/recent_emails.csv` (~31 emails)

**Pipeline:**
1. Load & inspect data
2. Preprocess email text (subject + body)
3. Fit BERTopic model
4. Explore & visualize discovered topics
5. Assign topic labels back to each email

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import pathlib
import warnings
warnings.filterwarnings('ignore')

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
from IPython.display import IFrame, display

# VS Code + classic notebook compatible renderer
# Falls back to writing HTML so charts always appear
pio.renderers.default = 'plotly_mimetype+notebook'

def show_fig(fig, name='plot'):
    """Render a Plotly figure inline, with an HTML IFrame fallback."""
    try:
        fig.show()
    except Exception:
        pass
    html_dir = pathlib.Path('../data/charts')
    html_dir.mkdir(parents=True, exist_ok=True)
    html_path = html_dir / f'{name}.html'
    fig.write_html(str(html_path), full_html=True)
    print(f'Chart saved → {html_path}')
    display(IFrame(src=str(html_path), width='100%', height=550))

# Reproducibility
SEED = 42
np.random.seed(SEED)

print('✅ All imports successful')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/recent_emails.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Quick preview of unique senders
print('Top senders:')
df['sender_name'].value_counts().head(10)

## 3. Preprocess Text

Combine `subject` and `body` for richer signal. Repeat the subject to give it more weight, then clean noise (URLs, unicode junk, excess whitespace).

In [ ]:
def clean_text(text: str, max_chars: int = 2000) -> str:
    """Strip noise and return a clean, truncated string."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'https?://\S+', '', text)                      # remove URLs
    text = re.sub(r'[\ufffd\u0000-\u001f\u007f]', ' ', text)     # remove control chars
    text = re.sub(r'\s+', ' ', text).strip()                      # collapse whitespace
    return text[:max_chars]


df['subject_clean'] = df['subject'].apply(clean_text)
df['body_clean']    = df['body'].apply(lambda x: clean_text(x, max_chars=2000))

# Repeat subject 3× to up-weight it relative to the body
df['doc'] = (df['subject_clean'] + ' ') * 3 + df['body_clean']

print('Sample doc (first email):')
print(df['doc'].iloc[0][:400])

## 4. Build & Fit BERTopic

Small corpus (~31 docs) → tune for sensitivity:
- **UMAP** `n_neighbors=5` preserves local structure
- **HDBSCAN** `min_cluster_size=2` avoids over-outlier-ing
- **all-MiniLM-L6-v2** — fast, high-quality sentence embeddings

In [ ]:
docs = df['doc'].tolist()

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=5,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    min_df=1,
    ngram_range=(1, 2)
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True
)

print('🚀 Fitting BERTopic...')
topics, probs = topic_model.fit_transform(docs)
print(f'\n✅ Done! Found {topic_model.get_topic_freq().shape[0] - 1} topic(s) (excluding outlier -1)')

## 5. Topic Overview

In [ ]:
freq = topic_model.get_topic_freq()
print('Topic frequencies:')
freq

In [ ]:
print('=== Topic Keywords ===')
for topic_id in freq['Topic'].tolist():
    label = '[Outliers / Unclustered]' if topic_id == -1 else f'[Topic {topic_id}]'
    print(f'\n{label}')
    words = topic_model.get_topic(topic_id)
    if words:
        print('  ', ', '.join([w for w, _ in words[:8]]))

## 6. Assign Topics Back to Emails

In [ ]:
df['topic_id']   = topics
df['topic_prob'] = probs

def topic_label(tid):
    if tid == -1:
        return 'Unclustered'
    words = topic_model.get_topic(tid)
    return f'T{tid}: {" / ".join([w for w, _ in words[:3]])}' if words else f'Topic {tid}'

df['topic_label'] = df['topic_id'].apply(topic_label)

print('Emails grouped by topic:\n')
for label, group in df.groupby('topic_label'):
    print(f'--- {label} ({len(group)} email{"s" if len(group)>1 else ""}) ---')
    for _, row in group.iterrows():
        print(f'  • {row["subject"][:90]}')
    print()

In [ ]:
cols = ['date', 'sender_name', 'subject', 'topic_id', 'topic_label', 'topic_prob']
df[cols].sort_values('topic_id')

## 7. Visualizations

> Charts render inline via `plotly_mimetype+notebook`. An HTML copy is also saved to `data/charts/` so you can open it in a browser if the inline view is blank.

In [ ]:
# --- 7a. Topic word-score bar chart ---
fig_bar = topic_model.visualize_barchart(top_n_topics=10, n_words=8)
show_fig(fig_bar, 'barchart')

In [ ]:
# --- 7b. Topic similarity heatmap ---
fig_heat = topic_model.visualize_heatmap()
show_fig(fig_heat, 'heatmap')

In [ ]:
# --- 7c. 2D document map ---
embeddings  = embedding_model.encode(docs, show_progress_bar=True)
reduced_2d  = UMAP(
    n_neighbors=5, n_components=2,
    min_dist=0.0, metric='cosine',
    random_state=SEED
).fit_transform(embeddings)

palette = [
    '#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A',
    '#19D3F3','#FF6692','#B6E880','#FF97FF','#FECB52'
]

fig_map = go.Figure()
for i, lbl in enumerate(df['topic_label'].unique()):
    mask   = df['topic_label'] == lbl
    color  = '#888' if 'Unclustered' in lbl else palette[i % len(palette)]
    fig_map.add_trace(go.Scatter(
        x=reduced_2d[mask, 0],
        y=reduced_2d[mask, 1],
        mode='markers+text',
        name=lbl,
        text=df[mask]['sender_name'].fillna('Unknown').tolist(),
        textposition='top center',
        marker=dict(size=11, color=color, opacity=0.85),
        hovertext=df[mask]['subject'].tolist(),
        hoverinfo='text+name'
    ))

fig_map.update_layout(
    title='📧 Email Document Map (2D UMAP)',
    template='plotly_dark',
    height=620,
    legend_title='Topic',
    xaxis_title='UMAP-1',
    yaxis_title='UMAP-2'
)
show_fig(fig_map, 'docmap')

## 8. Save Results

In [ ]:
out_path = '../data/emails_with_topics.csv'
df.drop(columns=['doc', 'subject_clean', 'body_clean'], errors='ignore').to_csv(out_path, index=False)
print(f'✅ Saved enriched CSV → {out_path}')
df[['subject', 'topic_id', 'topic_label', 'topic_prob']].head(10)